# Data cleanup

This script cleans up and forms a relationally-consistent subset of the ASDN data.  The main focus is on forming inter-entity relationships based on primary/foreign keys.  For the most part data that is not valid is simply discarded, though in some cases bad data is corrected or otherwise fixed.  The net result is that this training dataset is a subset of the original data in three ways:

- It contains only some of the original tables
- It contains only a few of the original columns
- It contains only as many rows as were practically possible to include

Regarding the last bullet, our goal is to be faithful to the original data, but with the recognition that our ultimate purpose is only to create a dataset supporting instruction.  If there is no easy and unambiguous way to fix data that is clearly invalid, we simply drop it.  As a result, needless to say, this cleaned-up data should not be relied upon for any actual analysis.

A foundational issue is that some files have bad character encodings, making them impossible to read in as is.  We replace the bad characters with U+FFFD REPLACEMENT CHARACTER and then, after filtering rows and columns down, fix any such characters that remain.

In [1]:
import re
import unicodedata

import pandas as pd

identifier_re = r"^[a-zA-Z0-9]+([-_./][a-zA-Z0-9]+)*$"

## Sites and species

The sites and species tables were hand-created from the README file.  The assertions match those in the SQL table definitions, plus a few more.

In [11]:
sites = pd.read_csv(
    "../../01_data/data-processed/sites.csv",
    dtype={
        "Code": "string",
        "Site_name": "string",
        "Location": "string",
        "Latitude": "float",
        "Longitude": "float",
        "Total_Study_Plot_Area": "float"
    }
)

assert all(sites.Code.notna())
assert len(sites.Code) == len(sites.Code.unique())
assert all(sites.Code.str.match(r"[a-z]{4}$"))
assert all(sites.Site_name.notna())
assert len(sites.Site_name) == len(sites.Site_name.unique())
assert all(sites.Location.notna())
assert all(sites.Latitude.notna())
assert all((sites.Latitude >= -90) & (sites.Latitude <= 90))
assert all(sites.Longitude.notna())
assert all((sites.Longitude >= -180) & (sites.Longitude <= 180))
assert all(sites.Total_Study_Plot_Area.notna())
assert all(sites.Total_Study_Plot_Area > 0)
assert all(~sites.duplicated(["Latitude", "Longitude"]))

species = pd.read_csv("../../01_data/data-processed/species.csv", dtype="string")

assert all(species.Code.notna())
assert len(species.Code) == len(species.Code.unique())
assert all(species.Code.str.match(r"[a-z]{4,5}$"))
assert all(species.Common_name.notna())
assert len(species.Common_name) == len(species.Common_name.unique())

## Bird nests

Cleaning up everything except `Observer` and `Clutch_max`, which we'll deal with further below.  `floatAge` is left in string form and uncorrected (other than to correlate it with `ageMethod == "float"`) to serve as a student exercise in data cleaning.

For `Observer`, at this point we just correct bad character encodings and collapse multiple observers.  Through researching the literature we uncovered the following names and created the following abbreviations for them.

|abbreviation|full name|
|---|---|
|`blaliberte`|Benoît Laliberté|
|`mgrahamsauve`|Maude Graham-Sauvé|
|`aguerettemontminy`|Alisa Guérette Montminy|

In [12]:
nests = pd.read_csv(
    "../../01_data/data-raw/ASDN_bird_nests.csv",
    encoding_errors="replace",
    usecols=[
        "Book_page",
        "Year",
        "Site",
        "Nest_ID",
        "Species",
        "Observer",
        "Date_found",
        "how_found",
        "Clutch_max",
        "floatAge",
        "ageMethod"
    ],
    dtype={
        "Book_page": "string",
        "Year": "Int64",
        "Site": "string",
        "Nest_ID": "string",
        "Species": "string",
        "Observer": "string",
        "how_found": "string",
        "Clutch_max": "Int64",
        "floatAge": "string",
        "ageMethod": "string"
    },
    parse_dates=["Date_found"],
    date_format="%d-%b-%y"
)
# 13,062 nests

# Remove nests with invalid species codes
nests = nests[nests.Species.isin(species.Code)]
# 13,056 nests

# Remove spurious spaces in nest IDs
nests.Nest_ID = nests.Nest_ID.str.replace(" ", "")

# Replace book page references that don't look like identifiers with NA
nests.loc[~nests.Book_page.str.match(identifier_re), "Book_page"] = pd.NA

# Remove nests without dates
nests = nests[nests.Date_found.notna()]
# 13,022 nests

# Collapse how_found values
nests.loc[nests.how_found == "rope  ", "how_found"] = "rope"
nests.loc[nests.how_found == "SYSTEMATIC SEARCH", "how_found"] = (
    "systematic search"
)

# Collapse ageMethod values
# N.B.: values "mean date" and "inc" are retained though they are not
# defined in the README.
nests.loc[nests.ageMethod == "none", "ageMethod"] = pd.NA
nests.loc[nests.ageMethod == "unknown", "ageMethod"] = pd.NA
nests.loc[nests.ageMethod == "lay ", "ageMethod"] = "lay"
nests.loc[nests.ageMethod == "float ", "ageMethod"] = "float"

# Adjust age values so that if there is a non-NA floatAge, then
# ageMethod = float.  (The converse is not true, a nest aged by
# floating may have an NA floatAge.)
nests.loc[nests.floatAge.notna(), "ageMethod"] = "float"

assert all(nests.Nest_ID.notna())
assert all(nests.Nest_ID.str.match(identifier_re))
assert len(nests.Nest_ID) == len(nests.Nest_ID.unique())
assert all(nests.Book_page.dropna().str.match(identifier_re))
assert all(nests.Year.notna())
assert all((nests.Year >= 1990) & (nests.Year <= 2015))
assert all(nests.Site.notna())
assert all(nests.Site.isin(sites.Code))
assert all(nests.Species.notna())
assert all(nests.Species.isin(species.Code))
assert all(nests.Date_found.notna())
assert all(nests.Date_found.dt.year == nests.Year)
assert all(nests.loc[nests.floatAge.notna(), "ageMethod"] == "float")

char_fixups = {
    "blalibert\ufffd": "blaliberte",
    "mgrahamsauv\ufffd": "mgrahamsauve",
    "agu\ufffdrettemontminy": "aguerettemontminy"
}

# Correct bad character encodings
for bad, good in char_fixups.items():
    nests.loc[nests.Observer == bad, "Observer"] = good

# Where there are multiple observers listed, just pick the first
nests.Observer = (
    nests.Observer.str.split(r"/|-|, ?|\+| and | & | &amp; ").str.get(0)
)

# Remove spurious spaces in observer abbreviations
nests.Observer = nests.Observer.str.replace(" ", "")

## Observers

The fundamental problem we deal with here is that there is no observer table, despite observer being an independent entity that is referred to from multiple other tables.  The README says observers are referred to by "first initial and last name (or initials) of person," but that's at best an informal practice.  There's a table, `Camp_staff`, that lists full names of observers, and this appears to be the only place where full names appear.  But that table contains no abbreviations, and thus there's no explicit linkage to that table from anywhere else.  In any case, the entity behind that table is not a person but rather a work stint, as people appear multiple times for different date ranges.  Our goal is:

- Create an `Observer` table that stores people, using abbreviation as the primary key
- Relabel `Camp_staff` as `Camp_assignments`
- Adjust `Camp_assignments` and `Bird_nests` to refer to `Observer` via foreign keys

Issues to be dealt with:

- Alternative name spellings/misspellings
- Unicode encoding problems
- Non-unique abbreviations

Limitations:

- We do not check that work stint dates are consistent with observation dates recorded by the observer
- We do not check that work stint date ranges are disjoint for given observer and site (in fact, there are some overlaps)

In [14]:
cs = pd.read_csv(
    "../../01_data/data-raw/ASDN_Camp_staff.csv",
    encoding="ISO-8859-1",
    dtype={
        "Year": "Int64",
        "Site": "string",
        "Name": "string"
    },
    parse_dates=["Start", "End"],
    date_format="%d-%b-%y"
)

# Remove entries for which dates are unknown
cs = cs[cs.Start.notna() & cs.End.notna()]

assert all(cs.Year.notna())
assert all((cs.Year >= 1990) & (cs.Year <= 2015))
assert all(cs.Site.notna())
assert all(cs.Site.isin(sites.Code))
assert all(cs.Name.notna())
assert all(cs.Start <= cs.End)
assert all((cs.Start.dt.year <= cs.Year) & (cs.Year <= cs.End.dt.year))

name_fixups = {
    "Alanah Kataluk-Primeau":   "Alannah Kataluk-Primeau",
    "Andrew Bankert":           "Andrew R. Bankert",
    "Andrew Doll":              "Andrew C. Doll",
    "Andy Johnson":             "Andrew S. Johnson",
    "Anja ":                    "Anja Unknown",
    "Brad Wilkinson":           "Bradley Wilkinson",
    "Brooke Hill":              "Brooke L. Hill",
    "Caitlin Daviis":           "Caitlin Davis",
    "Dave Mcgeachy":            "Dave McGeachy",
    "Dylan Kessler":            "Dylan Kesler",
    "Emilie D A'Stous":         "Émilie D'Astous",
    "Emilie D'Astrous":         "Émilie D'Astous",
    "Émilie D'astous":          "Émilie D'Astous",
    "Felicia Sanders ":         "Felicia Sanders",
    "Gennyne Mccune":           "Gennyne McCune",
    "Georgiy Pavlukov":         "Georgy Pavlukov",
    "Georgiy Pavlyukov":        "Georgy Pavlukov",
    "Johanna Perz ":            "Johanna Perz",
    "Laura Mckinnon":           "Laura McKinnon",
    "Madi Mcconnell":           "Madison McConnell",
    "Madison Mcconnell":        "Madison McConnell",
    "Mckenzie Mudge":           "McKenzie Mudge",
    "Meagan Mccloskey":         "Meagan McCloskey",
    "Metta Mcgarvey":           "Metta McGarvey",
    "Naomi Manin'T Veld":       "Naomi Man in 't Veld",
    "Richard Lanctot":          "Richard B. Lanctot",
    "Rick Lanctot (1St Visit)": "Richard B. Lanctot",
    "Rick Lanctot (1st visit)": "Richard B. Lanctot",
    "Rick Lanctot (2Nd Visit)": "Richard B. Lanctot",
    "Rick Lanctot (2nd visit)": "Richard B. Lanctot",
    "Ronan A Dugan":            "Ronan A. Dugan",
    "Sarah Saalfield":          "Sarah Saalfeld",
    "Sergey Vartanayn":         "Sergey Vartanyan",
    "Toby St Clair":            "Toby St. Clair",
    "Ty Donnelly":              "Tyrone Donnelly",
    "brendan higgins":          "Brendan Higgins",
    "claire montgomerie":       "Claire Montgomerie",
    "emily magnuson":           "Emily Magnuson",
    "ian davies":               "Ian Davies",
    "maureen correll":          "Maureen Correll",
    "scott freeman":            "Scott Freeman"
}

cs.Name = (
    cs.Name.apply(lambda name: name_fixups.get(name, name)).astype("string")
)

# Form canonical abbreviations.  Note that one abbreviation,
# `brobinson`, is not unique (it could refer to Brian Robinson or
# Bryce Robinson).  The goal is to abbreviate a name like
# "Jean-Claude B. St. Van Dämmé (Smith)" as `jcstvandamme`.

abbrevs = {}  # abbreviation => name

for name in cs.Name.unique():
    m = re.match(r"([\w-]+)(?: \w\.)? ([\w'. -]+)(?: \(\w+\))?$", name)
    first, last = m[1], m[2]
    abbrev = (
        "".join(n[0].lower() for n in first.split("-"))
        + re.sub("[^\\w]", "", last).lower()
    )
    # Remove diacritics
    abbrev = "".join(
        c for c in unicodedata.normalize("NFKD", abbrev)
        if not unicodedata.combining(c)
    )
    # Watch for duplicates
    if abbrev in abbrevs:
        if not type(abbrevs[abbrev]) is list:
            abbrevs[abbrev] = [abbrevs[abbrev]]
        abbrevs[abbrev].append(name)
    else:
        abbrevs[abbrev] = name

for abbrev, v in list(abbrevs.items()):
    if type(v) is list:
        del abbrevs[abbrev]
        for i in range(len(v)):
            abbrevs[abbrev + str(i+1)] = v[i]

Nest fixups... TBD

In [15]:
abbrev_fixups = {
    "lanctot": "rlanctot",
    "L.McKinnon": "lmckinnon"
}

for bad, good in abbrev_fixups.items():
    nests.loc[nests.Observer == bad, "Observer"] = good

todo = set()
for x in nests.loc[~nests.Observer.isin(abbrevs), "Observer"].unique():
    if type(x) is str:
        todo.add(x)

## Bird eggs

We require that `Year` and `Site` match between eggs and nests, but allow `Book_page` to differ on the assumption that egg and nest data can indeed come from different pages.  We do check, though, that `Book_page` is consistent across the eggs within a nest.

In [16]:
eggs = pd.read_csv(
    "../../01_data/data-raw/ASDN_bird_eggs.csv",
    usecols=[
        "Book_page",
        "Year",
        "Site",
        "Nest_ID",
        "Egg_num",
        "Length",
        "Width"
    ],
    dtype={
        "Book_page": "string",
        "Year": "Int64",
        "Site": "string",
        "Nest_ID": "string",
        "Egg_num": "Int64",
        "Length": "float",
        "Width": "float"
    }
)
# 26,564 eggs

# Select only those eggs with non-NA primary keys (will validate keys later)
eggs = eggs[eggs.Nest_ID.notna() & eggs.Egg_num.notna()]
# 25,901 eggs

# Limit eggs to where a nest exists.  It's really unfortunate to lose
# so many eggs (almost 30%) but there's no clear correction to be
# applied.  A nest ID that's not in the nests table just seems
# missing.
eggs = eggs[eggs.Nest_ID.isin(nests.Nest_ID.unique())]
# 18,702 eggs

# Limit eggs to where egg numbers form a consecutive sequence 1, 2, 3, ...
def completeness_test(group):
    return set(group.Egg_num) == set(en+1 for en in range(len(group.Egg_num)))
is_complete = (
    eggs.groupby("Nest_ID").apply(completeness_test, include_groups=False)
)
eggs = eggs[eggs.Nest_ID.isin(is_complete[is_complete].index)]
# 18,612 eggs

# Limit eggs to where Book_page has a consistent value across the nest
is_consistent = eggs.groupby("Nest_ID").Book_page.nunique() <= 1
eggs = eggs[eggs.Nest_ID.isin(is_consistent[is_consistent].index)]
# 18,608 eggs

assert all(eggs.Book_page.dropna().str.match(identifier_re))
assert all(eggs.Year.notna())
assert all(eggs.Site.notna())
joined = eggs.merge(nests, on="Nest_ID")
assert all(joined.Year_x == joined.Year_y)
assert all(joined.Site_x == joined.Site_y)
assert all(eggs.Length.notna())
assert all(eggs.Width.notna())
assert all((eggs.Length > 0) & (eggs.Length < 100))
assert all((eggs.Width > 0) & (eggs.Width < 100))

# Back to nests, ensure Clutch_max is greater than or equal to the
# number of eggs measured
fixups = (
    joined[joined.Clutch_max < joined.Egg_num].groupby("Nest_ID").Egg_num.max()
)
for nest_id in fixups.index:
    nests.loc[nests.Nest_ID == nest_id, "Clutch_max"] = fixups[nest_id]

## Output

In [17]:
nests.to_csv("../../01_data/data-processed/bird-nests.csv", index=False)
cs.to_csv("../../01_data/data-processed/camp-assignments.csv", index=False)
eggs.to_csv("../../01_data/data-processed/bird-eggs.csv", index=False)

observers = pd.DataFrame(
    list(abbrevs.items()),
    columns=["Abbreviation", "Name"]
)
observers.to_csv("../../01_data/data-processed/observers.csv", index=False)